### 1
(Python/R): Для белка P04637 (p53) получите список его функций (features)

In [1]:
import requests
import json

def fetch_protein_characteristics(protein_id, limit=None):
    api_endpoint = f"https://www.ebi.ac.uk/proteins/api/proteins/{protein_id}"
    
    server_response = requests.get(api_endpoint, headers={"Accept": "application/json"})
    
    if server_response.status_code == 200:
        protein_data = server_response.json()
        characteristics_list = protein_data.get('features', [])
        
        output_filename = f"{protein_id}_characteristics.json"
        with open(output_filename, "w") as output_file:
            json.dump(characteristics_list, output_file, indent=2)
        
        if limit is not None and len(characteristics_list) > limit:
            print(f"Обнаружено характеристик: {len(characteristics_list)}. Отображаются первые {limit}.")
            characteristics_list = characteristics_list[:limit]
            
        return characteristics_list
    else:
        print(f"Не удалось загрузить данные: {server_response.status_code}")
        return None

p53_characteristics = fetch_protein_characteristics("P04637", limit=10)

if p53_characteristics:
    print("\nХарактеристики белка p53 (P04637):")
    print("=" * 70)
    for index, characteristic in enumerate(p53_characteristics, 1):
        char_type = characteristic.get('type', 'Не указано')
        char_description = characteristic.get('description', 'Описание отсутствует')
        start_pos = characteristic.get('begin', 'Не указано')
        end_pos = characteristic.get('end', 'Не указано')
        
        print(f"{index:2d}. {char_type:12} | {char_description[:35]:35} | Позиции: {start_pos}-{end_pos}")
    print("=" * 70)
    print(f"Все характеристики сохранены в файле P04637_characteristics.json")
else:
    print("Загрузка характеристик белка не удалась")

Обнаружено характеристик: 1518. Отображаются первые 10.

Характеристики белка p53 (P04637):
 1. CHAIN        | Cellular tumor antigen p53          | Позиции: 1-393
 2. DNA_BIND     |                                     | Позиции: 102-292
 3. REGION       | Interaction with CCAR2              | Позиции: 1-320
 4. REGION       | Interaction with HRMT1L2            | Позиции: 1-83
 5. REGION       | Transcription activation (acidic)   | Позиции: 1-44
 6. REGION       | Disordered                          | Позиции: 50-96
 7. REGION       | Interaction with WWOX               | Позиции: 66-110
 8. REGION       | Interaction with HIPK1              | Позиции: 100-370
 9. REGION       | Required for interaction with ZNF38 | Позиции: 100-300
10. REGION       | Required for interaction with FBXO4 | Позиции: 113-236
Все характеристики сохранены в файле P04637_characteristics.json


### 2 
(Python/R): Получите последовательности белков P05067 (APP) и P10636 (Tau). Сравните их длины. Какой белок длиннее и на сколько аминокислот?


In [3]:
import requests

def retrieve_amino_acid_chain(uniprot_id):
    api_url = "https://www.ebi.ac.uk/proteins/api/proteins"
    query_params = {'accession': uniprot_id}
    
    api_response = requests.get(api_url, params=query_params, headers={"Accept": "application/json"})
    
    if api_response.status_code == 200:
        protein_info = api_response.json()
        if isinstance(protein_info, list) and protein_info:
            return protein_info[0]['sequence']['sequence']
    print(f"Не удалось получить последовательность для {uniprot_id}: код ошибки {api_response.status_code}")
    return None

amyloid_precursor = retrieve_amino_acid_chain("P05067")
microtubule_binding = retrieve_amino_acid_chain("P10636")

if amyloid_precursor and microtubule_binding:
    amyloid_length = len(amyloid_precursor)
    tau_length = len(microtubule_binding)
    
    print(f"Аминокислот в белке APP (P05067): {amyloid_length}")
    print(f"Аминокислот в белке Tau (P10636): {tau_length}")
    
    if amyloid_length > tau_length:
        difference = amyloid_length - tau_length
        print(f"APP содержит больше аминокислот на {difference}")
    elif tau_length > amyloid_length:
        difference = tau_length - amyloid_length
        print(f"Tau содержит больше аминокислот на {difference}")
    else:
        print("Оба белка имеют идентичное количество аминокислот")

Аминокислот в белке APP (P05067): 770
Аминокислот в белке Tau (P10636): 758
APP содержит больше аминокислот на 12


### 3
(Python/R): Получите данные о 5 различных белках (на ваш выбор) через API, используя их accession IDs. Сохраните их entryName и длину последовательности в таблицу (data frame в R/Pandas)

In [8]:
!pip install pandas

  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl (11.0 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ------------- -------------------------- 1/3 [tzdata]
   ------------- -------------------------- 1/3 [tzdata]
   ------------- -------------------------- 1/3 [tzdata]
   ------------- -------------------------- 1/3 [tzdata]
   -----


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import requests
import pandas as pd

protein_ids = ["P04637", "P05067", "P10636", "P00738", "P02768"]
protein_collection = []

for protein_id in protein_ids:
    api_url = "https://www.ebi.ac.uk/proteins/api/proteins"
    query_parameters = {'accession': protein_id}
    
    server_response = requests.get(api_url, params=query_parameters, headers={"Accept": "application/json"})
    
    if server_response.status_code == 200:
        json_data = server_response.json()
        if isinstance(json_data, list) and json_data:
            protein_identifier = json_data[0]['id']
            aa_chain_length = len(json_data[0]['sequence']['sequence'])
            protein_collection.append({
                'ProteinID': protein_id,
                'EntryName': protein_identifier,
                'ChainLength': aa_chain_length
            })
    else:
        print(f"Сбой загрузки данных для {protein_id}: статус {server_response.status_code}")

protein_table = pd.DataFrame(protein_collection)
print("Результаты анализа протеинов:")
print(protein_table)
protein_table.to_csv("protein_analysis.csv", index=False)

Результаты анализа протеинов:
  ProteinID   EntryName  ChainLength
0    P04637   P53_HUMAN          393
1    P05067    A4_HUMAN          770
2    P10636   TAU_HUMAN          758
3    P00738   HPT_HUMAN          406
4    P02768  ALBU_HUMAN          609


### 4
(Python/R): Используя API PubMed, найдите 10 самых последних (сортировка по дате) статей по запросу "COVID-19 variants". Выведите их ID и заголовки.

In [ ]:
import requests
from xml.etree import ElementTree as ET

def fetch_pubmed_articles(search_term, result_limit=10):
    search_endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        'db': 'pubmed',
        'term': search_term,
        'retmode': 'json',
        'retmax': result_limit,
        'sort': 'pubdate'
    }
    
    api_response = requests.get(search_endpoint, params=search_params)
    
    if api_response.status_code == 200:
        result_data = api_response.json()
        if 'esearchresult' in result_data and 'idlist' in result_data['esearchresult']:
            return result_data['esearchresult']['idlist']
    print(f"Поисковый запрос не выполнен: код {api_response.status_code}")
    return []

def retrieve_publication_details(publication_ids):
    fetch_endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {
        'db': 'pubmed',
        'id': ','.join(publication_ids),
        'retmode': 'xml'
    }
    
    api_response = requests.get(fetch_endpoint, params=fetch_params)
    
    if api_response.status_code == 200:
        return api_response.text
    print(f"Не удалось загрузить данные публикаций: {api_response.status_code}")
    return None

found_publications = fetch_pubmed_articles("COVID-19 variants", 10)

if found_publications:
    print("Идентификаторы обнаруженных публикаций:")
    print(found_publications)
    publications_xml = retrieve_publication_details(found_publications)
    
    if publications_xml:
        xml_root = ET.fromstring(publications_xml)
        
        print("\nДетали публикаций:")
        for index, pub_id in enumerate(found_publications):
            try:
                pub_element = xml_root.findall(".//PubmedArticle")[index]
                publication_title = pub_element.find(".//ArticleTitle").text
                
                print(f"Публикация #{index+1}: {pub_id}")
                print(f"Название: {publication_title}\n")
            except Exception as error:
                print(f"Ошибка обработки публикации {pub_id}: {error}")

Идентификаторы обнаруженных публикаций:
['41196059', '41194062', '41193798', '41193785', '41191496', '41191293', '41190654', '41188882', '41188857', '41188743']

Детали публикаций:
Публикация #1: 41196059
Название: Mouse-adapted SARS-CoV-2 Omicron BA.5 infection induces post-acute lung fibrosis in BALB/c mice.

Публикация #2: 41194062
Название: Dutch participatory surveillance framework for evaluating evolutionary changes on SARS-CoV-2 affecting rapid diagnostic test sensitivity in 2022 - 2023.

Публикация #3: 41193798
Название: Assessing phylogenetic confidence at pandemic scales.

Публикация #4: 41193785
Название: Organoids in respiratory virus research: advances and perspectives.

Публикация #5: 41191496
Название: Fuel-Free Rolosense: Viral Sensing Using Diffusional Particle Tracking.

Публикация #6: 41191293
Название: "Just be honest with us": A qualitative analysis of Canadians' public trust during the COVID-19 pandemic.

Публикация #7: 41190654
Название: Tradeoffs in viral fitnes

### 5
(Python): Напишите функцию get_protein_name(accession), которая принимает на вход accession ID белка и возвращает его название (entryName). Обработайте случай, если белок не найден.

In [11]:
import requests

def fetch_protein_identifier(uniprot_id):
    api_endpoint = "https://www.ebi.ac.uk/proteins/api/proteins"
    query_params = {'accession': uniprot_id}
    
    api_response = requests.get(api_endpoint, params=query_params, headers={"Accept": "application/json"})
    
    if api_response.status_code == 200:
        response_data = api_response.json()
        if isinstance(response_data, list) and response_data:
            return response_data[0]['id']
        return f"Идентификатор {uniprot_id} не обнаружен в базе данных"
    return f"Сбой подключения к серверу: код {api_response.status_code}"
print("P04637 соответствует:", fetch_protein_identifier("P04637"))
print("P10636 соответствует:", fetch_protein_identifier("P10636"))
print("Некорректный ID:", fetch_protein_identifier("INVALID_ID"))

P04637 соответствует: P53_HUMAN
P10636 соответствует: TAU_HUMAN
Некорректный ID: Сбой подключения к серверу: код 400


### 6
(Python): Прочитайте из файла accessions.txt (нужно создать) список из 3-5 accession IDs. Для каждого ID получите из Proteins API и запишите в новый файл его название и организм-источник.

In [ ]:
import requests

def get_protein_info(accession):
    url = "https://www.ebi.ac.uk/proteins/api/proteins"
    params = {'accession': accession}
    
    response = requests.get(url, params=params, headers={"Accept": "application/json"})
    
    if response.status_code == 200:
        data = response.json()
        if isinstance(data, list) and len(data) > 0:
            protein = data[0]
            entry_name = protein['id']
            organism = protein['organism']['names'][0]['value']
            return entry_name, organism
    return None, None

with open("accessions.txt", "w") as f:
    f.write("P04637\nP05067\nP10636\nP00738\nP02768")
with open("accessions.txt", "r") as f:
    accessions = [line.strip() for line in f.readlines()]
with open("protein_info.txt", "w") as output_file:
    output_file.write("Accession\tEntryName\tOrganism\n")
    for acc in accessions:
        entry_name, organism = get_protein_info(acc)
        if entry_name and organism:
            output_file.write(f"{acc}\t{entry_name}\t{organism}\n")

Информация о белках сохранена в файл protein_info.txt


### 7
(Python/R): Для найденных в задании 4 статей получите более подробную информацию, используя Eutils API и ID статей (конечная точка efetch.fcgi). Извлеките имена авторов для каждой статьи.

In [ ]:
import requests
import xml.etree.ElementTree as ET

def fetch_pubmed_articles(search_query, max_results=10):
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    parameters = {
        'db': 'pubmed',
        'term': search_query,
        'retmode': 'json',
        'retmax': max_results,
        'sort': 'pubdate'
    }
    
    response = requests.get(search_url, params=parameters)
    
    if response.status_code == 200:
        data = response.json()
        if 'esearchresult' in data and 'idlist' in data['esearchresult']:
            return data['esearchresult']['idlist']
    print(f"Ошибка поиска: {response.status_code}")
    return []

def fetch_article_details(article_ids):
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    parameters = {
        'db': 'pubmed',
        'id': ','.join(article_ids),
        'retmode': 'xml'
    }
    
    response = requests.get(fetch_url, params=parameters)
    
    if response.status_code == 200:
        return response.text
    print(f"Ошибка загрузки статей: {response.status_code}")
    return None

def extract_authors_info(xml_content, article_ids):
    root = ET.fromstring(xml_content)
    
    print("\nАвторы научных статей:")
    for i, article_id in enumerate(article_ids):
        try:
            article = root.findall(".//PubmedArticle")[i]
            authors = []
            author_elements = article.findall(".//Author")
            for author in author_elements:
                last_name_elem = author.find("LastName")
                first_name_elem = author.find("ForeName")
                
                if last_name_elem is not None and first_name_elem is not None:
                    authors.append(f"{first_name_elem.text} {last_name_elem.text}")
            
            print(f"Статья ID: {article_id}")
            if authors:
                print("Авторы:", " | ".join(authors))
            else:
                print("Авторы не указаны")
            print("=" * 70)
            
        except Exception as e:
            print(f"Ошибка обработки статьи {article_id}: {e}")
            
article_ids = fetch_pubmed_articles("COVID-19 variants", 5)

if article_ids:
    print("Найденные ID статей:")
    print(article_ids)
    
    xml_data = fetch_article_details(article_ids)
    
    if xml_data:
        extract_authors_info(xml_data, article_ids)

Найденные ID статей:
['41196059', '41194062', '41193798', '41193785', '41191496']

Авторы научных статей:
Статья ID: 41196059
Авторы: John M Powers | Sarah R Leist | Naveenchandra Suryadevara | Seth J Zost | Elad Binshtein | Anfal Abdelgadir | Michael L Mallory | Caitlin E Edwards | Kendra L Gully | Miranda L Hubbard | Mark R Zweigart | Alexis B Bailey | Timothy P Sheahan | James E Crowe | Stephanie A Montgomery | Jack R Harkema | Ralph S Baric
Статья ID: 41194062
Авторы: Eva Kozanli | Wanda Han | Tara Smit | Jordy de Bakker | Mansoer Elahi | Ryanne Jaarsma | Gesa Carstens | Albert Jan van Hoek | Dirk Eggink
Статья ID: 41193798
Авторы: Nicola De Maio | Nhan Ly-Trong | Samuel Martin | Bui Quang Minh | Nick Goldman
Статья ID: 41193785
Авторы: Xingling Li | Haiqing Xiao | Ming Zhou | Chuanlai Yang | Xinyi Yang | Tong Cheng | Lunzhi Yuan | Ningshao Xia
Статья ID: 41191496
Авторы: Selma Piranej | Krista Jackson | Luona Zhang | Jacob Kæstel-Hansen | Frank Sommerhage | David DeRoo | Nikos S H

### 8
(Bash): Напишите команду curl, которая получает данные о белке, выполняет поиск по слову "phosphorylation" (либо другое на ваш выбор) в ответе и выводит строки, содержащие это слово.

In [26]:
#!curl -s -H "Accept: application/json" "https://www.ebi.ac.uk/proteins/api/proteins?accession=P10636" | grep -i "phosphorylation" --color=always

### 9
Белок Q8WZ42 (Titin):  получить последовательность и разбить на строки по 80 символов

In [27]:
import requests

def fetch_protein_sequence(uniprot_id):
    api_url = f"https://www.ebi.ac.uk/proteins/api/proteins?accession={uniprot_id}"
    request_headers = {"Accept": "application/json"}
    
    api_response = requests.get(api_url, headers=request_headers)
    if api_response.status_code == 200:
        protein_data = api_response.json()
        
        if isinstance(protein_data, list) and protein_data:
            primary_record = protein_data[0]
            
            if "sequence" in primary_record and isinstance(primary_record["sequence"], str):
                aa_sequence = primary_record["sequence"]
            elif "sequence" in primary_record and isinstance(primary_record["sequence"], dict) and "sequence" in primary_record["sequence"]:
                aa_sequence = primary_record["sequence"]["sequence"]
            else:
                fasta_endpoint = f"https://www.ebi.ac.uk/proteins/api/proteins/{uniprot_id}.fasta"
                fasta_data = requests.get(fasta_endpoint)
                if fasta_data.status_code == 200:
                    fasta_lines = fasta_data.text.strip().split('\n')
                    aa_sequence = ''.join(fasta_lines[1:])
                else:
                    print(f"FASTA endpoint unavailable. Status: {fasta_data.status_code}")
                    return None
            
            formatted_aa_chain = "\n".join([aa_sequence[i:i+80] for i in range(0, len(aa_sequence), 80)])
            
            with open("titin_aa_sequence.txt", "w") as output_file:
                output_file.write(formatted_aa_chain)
            print("Amino acid sequence saved to titin_aa_sequence.txt")
            print(f"Sequence length: {len(aa_sequence)} residues")
            return aa_sequence
        else:
            print("Unexpected API response format")
            return None
    else:
        print(f"API request failed: {api_response.status_code}")
        print(api_response.text)
        return None
protein_sequence = fetch_protein_sequence("Q8WZ42")

Amino acid sequence saved to titin_aa_sequence.txt
Sequence length: 34350 residues


### 10
(R): Получите информацию о 10 белках разной длины. Постройте в R столбчатую диаграмму (barplot), отображающую длину каждого белка.

In [ ]:
library(httr)
library(jsonlite)
library(ggplot2)

protein_ids <- c("P10636", "Q8WZ42", "P05067", "P04637", "P06400", 
                "P02768", "P69905", "P02649", "P02751", "P02787")

protein_labels <- character(length(protein_ids))
sequence_sizes <- numeric(length(protein_ids))

for (i in seq_along(protein_ids)) {
    protein_id <- protein_ids[i]
    api_endpoint <- paste0("https://www.ebi.ac.uk/proteins/api/proteins?accession=", protein_id)
    
    server_response <- GET(api_endpoint, add_headers(Accept = "application/json"))
    
    if (status_code(server_response) == 200) {
        json_data <- content(server_response, "parsed")
        protein_title <- tryCatch({
            json_data[[1]]$protein$recommendedName$fullName$value
        }, error = function(e) {
            "Unidentified protein"
        })
        aa_chain <- json_data[[1]]$sequence
        chain_length <- nchar(aa_chain)
        
        protein_labels[i] <- protein_title
        sequence_sizes[i] <- chain_length
    } else {
        cat(sprintf("Data retrieval failed for %s: status %d\n", protein_id, status_code(server_response)))
        protein_labels[i] <- paste("Failed:", protein_id)
        sequence_sizes[i] <- 0
    }
}
protein_data <- data.frame(
    Protein = protein_labels,
    Length = sequence_sizes,
    ID = protein_ids
)
protein_data$ShortName <- ifelse(
    nchar(protein_data$Protein) > 12,
    paste0(substr(protein_data$Protein, 1, 12), "..."),
    protein_data$Protein
)
ggplot(protein_data, aes(x = reorder(ShortName, -Length), y = Length, fill = ID)) +
    geom_bar(stat = "identity", alpha = 0.8) +
    scale_fill_viridis_d() +
    labs(
        title = "Сравнение размеров белковых последовательностей",
        x = "Белки",
        y = "Аминокислотные остатки"
    ) +
    theme_minimal() +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
        plot.title = element_text(hjust = 0.5, size = 14),
        legend.position = "none"
    ) +
    coord_flip()

ModuleNotFoundError: No module named 'matplotlib'

### 11
(Python/R): Напишите скрипт, который для заданного гена (например, APOE) находит белки у человека, получает их ID, названия и ссылки на соответствующие записи в UniProt. Сохраните результат в CSV-файл.


In [ ]:
library(httr)
library(jsonlite)
library(dplyr)
library(readr)

gene <- "APOE"
organism <- "Homo+sapiens"
url <- paste0("https://www.ebi.ac.uk/proteins/api/proteins?gene=", gene, "&organism=", organism, "&size=100")

response <- GET(url, add_headers(Accept = "application/json"))

if (status_code(response) == 200) {
    data <- content(response, "parsed", encoding = "UTF-8")
    results <- data.frame(
        UniProtID = character(),
        ProteinName = character(),
        UniProtLink = character(),
        stringsAsFactors = FALSE
    )
    for (protein in data) {
        accession <- protein$accession
        protein_name <- tryCatch({
            protein$protein$recommendedName$fullName$value
        }, error = function(e) {
            "N/A"
        })
        uniprot_link <- paste0("https://www.uniprot.org/uniprot/", accession)
        
        results <- rbind(results, data.frame(
            UniProtID = accession,
            ProteinName = protein_name,
            UniProtLink = uniprot_link,
            stringsAsFactors = FALSE
        ))
    }
    write_csv(results, "apoe_proteins.csv")
    
    cat(sprintf("Обнаружено и сохранено %d белков в apoe_proteins.csv\n", nrow(results)))
    
} else {
    cat(sprintf("Ошибка при получении данных: %d\n", status_code(response)))
}

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1294459634.py, line 13)

### 12
(Python): Ваш коллега написал код, который в цикле делает 100 запросов к API (по 1 белку). Перепишите код, чтобы сделать всего 1 запрос, получающий данные сразу по 100 белкам (используйте соответствующий параметр в Proteins API)

In [30]:
import requests

protein_identifiers = ["P10636-1", "P10636-2", "P05067", "Q8WZ42"] * 25  # 100 записей

api_endpoint = "https://www.ebi.ac.uk/proteins/api/proteins"
query_parameters = {"accession": protein_identifiers, "size": len(protein_identifiers)}
request_headers = {"Accept": "application/json"}

api_response = requests.get(api_endpoint, params=query_parameters, headers=request_headers)

if api_response.status_code == 200:
    protein_collection = api_response.json()
    print(f"Данные успешно загружены для {len(protein_collection)} протеинов")
else:
    print(f"Сбой выполнения запроса: {api_response.status_code}")
    print(api_response.text)

Данные успешно загружены для 1 протеинов


In [31]:
str(protein_collection)[:100]

"[{'accession': 'P10636-1', 'id': 'TAU_HUMAN', 'proteinExistence': 'Evidence at protein level', 'info"

### 13
(R): Сделайте запрос к Proteins API, чтобы получить данные о белке в формате FASTA. Сохраните результат в файл с расширением .fasta

In [ ]:
library(httr)

uniprot_id <- "P18636"
fasta_endpoint <- paste0("https://www.ebi.ac.uk/proteins/api/proteins/", uniprot_id, ".fasta")

tryCatch({
    server_response <- GET(fasta_endpoint)
    
    if (status_code(server_response) == 200) {
        writeLines(content(server_response, "text"), "protein_sequence.fasta")
        cat("FASTA последовательность сохранена в protein_sequence.fasta\n")
    } else {
        cat(paste("Сбой загрузки: код ответа", status_code(server_response), "\n"))
        cat(paste("Детали:", content(server_response, "text"), "\n"))
    }
}, error = function(e) {
    cat(paste("Ошибка сетевого соединения:", e$message, "\n"))
})

FASTA последовательность сохранена в protein_sequence.fasta


### 14
Создайте сценарий, который целенаправленно вызывает ошибку 404 при работе с API, и корректно их обрабатывает с выводом информативных сообщений.

In [35]:
import requests

def retrieve_protein_info(uniprot_id):
    api_url = f"https://www.ebi.ac.uk/proteins/api/proteins/{uniprot_id}"
    request_headers = {"Accept": "application/json"}
    
    try:
        api_response = requests.get(api_url, headers=request_headers)
        api_response.raise_for_status()
        return api_response.json()
    except requests.exceptions.HTTPError as http_error:
        if api_response.status_code == 404:
            print(f"Ресурс не обнаружен: идентификатор '{uniprot_id}' отсутствует в UniProt.")
            print("Убедитесь в корректности идентификатора или наличии записи в базе данных.")
    except Exception as general_error:
        print(f"Неожиданная ошибка при выполнении запроса: {general_error}")
    
    return None

# Демонстрация работы с некорректным идентификатором
incorrect_id = "INVALID_ID_12345"
protein_result = retrieve_protein_info(incorrect_id)

# Демонстрация работы с корректным идентификатором
correct_id = "P10636"
protein_result = retrieve_protein_info(correct_id)
if protein_result:
    print(f"Успешно загружена информация по протеину: {protein_result['protein']}")

Успешно загружена информация по протеину: {'recommendedName': {'fullName': {'value': 'Microtubule-associated protein tau', 'evidences': [{'code': 'ECO:0000305'}]}}, 'alternativeName': [{'fullName': {'value': 'Neurofibrillary tangle protein'}}, {'fullName': {'value': 'Paired helical filament-tau'}, 'shortName': [{'value': 'PHF-tau'}]}]}


### 15
Напишите скрипт, который принимает название гена, ищет по нему белки у человека, а затем для первого найденного белка ищет статьи в PubMed, упоминающие его ген. Выведите ID статей.

In [36]:
import requests
import json

def fetch_human_proteins_by_gene(gene_symbol):
    """Получает протеины человека по символу гена"""
    api_endpoint = "https://www.ebi.ac.uk/proteins/api/proteins"
    query_params = {
        'gene': gene_symbol,
        'organism': 'Homo sapiens',
        'size': 5
    }
    request_headers = {"Accept": "application/json"}
    
    api_response = requests.get(api_endpoint, params=query_params, headers=request_headers)
    
    if api_response.status_code != 200:
        print(f"Сбой при запросе протеинов: {api_response.status_code}")
        print(api_response.text)
        return None
    
    try:
        response_data = api_response.json()
        if not response_data:
            print(f"Протеины для гена {gene_symbol} не обнаружены")
            return None
        return response_data
    except json.JSONDecodeError:
        print("Некорректный JSON в ответе Proteins API")
        print("Фрагмент ответа:", api_response.text[:200] + "...")
        return None

def query_pubmed_by_gene(gene_symbol):
    """Выполняет поиск публикаций в PubMed по символу гена"""
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        'db': 'pubmed',
        'term': f'"{gene_symbol}"[Gene] AND "human"[Organism]',
        'retmode': 'json',
        'retmax': 10
    }
    
    api_response = requests.get(search_url, params=search_params)
    
    if api_response.status_code != 200:
        print(f"Ошибка поиска в PubMed: {api_response.status_code}")
        print(api_response.text)
        return []
    
    try:
        result_data = api_response.json()
        if 'esearchresult' not in result_data or 'idlist' not in result_data['esearchresult']:
            print("Не удалось извлечь идентификаторы публикаций из PubMed")
            return []
        
        return result_data['esearchresult']['idlist']
    except json.JSONDecodeError:
        print("Ошибка парсинга JSON от PubMed")
        print("Фрагмент ответа:", api_response.text[:200] + "...")
        return []

def execute_gene_analysis(gene_symbol):
    """Выполняет комплексный анализ по указанному гену"""
    
    print(f"Анализ протеинов человека для гена {gene_symbol}...")
    protein_results = fetch_human_proteins_by_gene(gene_symbol)
    
    if not protein_results:
        return
    
    primary_protein = protein_results[0]
    protein_accession = primary_protein.get('accession', '')
    protein_entry = primary_protein.get('entryName', '')
    gene_info = primary_protein.get('gene', [{}])[0] if primary_protein.get('gene') else {}
    identified_gene = gene_info.get('name', {}).get('value', gene_symbol) if gene_info else gene_symbol
    
    print(f"\nОсновной обнаруженный протеин:")
    print(f"  UniProt ID: {protein_accession}")
    print(f"  Идентификатор: {protein_entry}")
    print(f"  Символ гена: {identified_gene}")
    
    print(f"\nПоиск научных публикаций для '{identified_gene}'...")
    publication_ids = query_pubmed_by_gene(identified_gene)
    
    if publication_ids:
        print(f"\nОбнаружено {len(publication_ids)} публикаций (идентификаторы):")
        for idx, pub_id in enumerate(publication_ids, 1):
            print(f"{idx}. {pub_id}")
        
        print("\nПрямые ссылки на публикации:")
        for pub_id in publication_ids[:5]:
            print(f"https://pubmed.ncbi.nlm.nih.gov/{pub_id}/")
    else:
        print("Публикации не обнаружены")

execute_gene_analysis("TP53")

Анализ протеинов человека для гена TP53...

Основной обнаруженный протеин:
  UniProt ID: A0A087WT22
  Идентификатор: 
  Символ гена: TP53

Поиск научных публикаций для 'TP53'...

Обнаружено 10 публикаций (идентификаторы):
1. 41196374
2. 41196329
3. 41194226
4. 41194193
5. 41194031
6. 41193733
7. 41193463
8. 41191529
9. 41190720
10. 41189077

Прямые ссылки на публикации:
https://pubmed.ncbi.nlm.nih.gov/41196374/
https://pubmed.ncbi.nlm.nih.gov/41196329/
https://pubmed.ncbi.nlm.nih.gov/41194226/
https://pubmed.ncbi.nlm.nih.gov/41194193/
https://pubmed.ncbi.nlm.nih.gov/41194031/


### 16
(Bash): Напишите bash-скрипт, который с помощью curl и jq (если установлен) извлекает accession ID из заранее сохраненного JSON-файла с данными белка и выводит его.

In [40]:
!jq -r '.accession' protein_data.json

"jq" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


### 17
Выберите любой интересующий вас белок или ген. Самостоятельно придумайте и выполните 3 различных запроса к Proteins API или PubMed, которые дадут о нем новую информацию. Опишите, что вы узнали.

возьмем Tau-белок (P10636), который играет ключевую роль в нейродегенеративных заболеваниях, включая болезнь Альцгеймера

Запрос 1: Анализ посттрансляционных модификаций

In [ ]:
import requests

url = "https://www.ebi.ac.uk/proteins/api/proteins/P10636"
response = requests.get(url, headers={"Accept": "application/json"})

if response.status_code == 200:
    data = response.json()
    
    print("Посттрансляционные модификации Tau-белка:")
    print("=" * 60)
    
    modifications = []
    for feature in data.get('features', []):
        if feature.get('type') in ['MOD_RES', 'LIPID', 'CARBOHYD']:
            mod_desc = feature.get('description', 'Нет описания')
            position = feature.get('begin', 'N/A')
            modifications.append({
                'type': feature['type'],
                'description': mod_desc,
                'position': position
            })
    mod_types = {}
    for mod in modifications:
        mod_type = mod['type']
        if mod_type not in mod_types:
            mod_types[mod_type] = []
        mod_types[mod_type].append(mod)
    
    for mod_type, mods in mod_types.items():
        print(f"\n{mod_type}:")
        for mod in mods[:3]: 
            print(f"  Позиция {mod['position']}: {mod['description']}")

Посттрансляционные модификации Tau-белка:

MOD_RES:
  Позиция 2: N-acetylalanine
  Позиция 18: Phosphotyrosine; by FYN
  Позиция 29: Phosphotyrosine

CARBOHYD:
  Позиция 87: N-linked (Glc) (glycation) lysine; in PHF-tau; in vitro
  Позиция 383: N-linked (Glc) (glycation) lysine; in PHF-tau; in vitro
  Позиция 467: N-linked (Glc) (glycation) lysine; in PHF-tau; in vitro


Tau-белок подвергается множественным регуляторным модификациям: фосфорилирование тирозинов указывает на его роль в синаптической передаче, а специфическое гликозилирование лизинов напрямую связано с патологической агрегацией при болезни Альцгеймера. Эти конкретные модификации служат потенциальными биомаркерами для ранней диагностики и мишенями для терапевтического вмешательства при нейродегенеративных заболеваниях.

Запрос 2: Распределение фосфо-сайтов Tau по структурным доменам

In [47]:
import requests

# Анализ сайтов фосфорилирования Tau
url = "https://www.ebi.ac.uk/proteins/api/features/P10636"
response = requests.get(url, headers={"Accept": "application/json"})

if response.status_code == 200:
    data = response.json()
    
    print("Сайты фосфорилирования Tau-белка:")
    print("=" * 50)
    
    phosphorylation_sites = []
    for feature in data.get('features', []):
        if feature.get('type') == 'MOD_RES':
            description = feature.get('description', '').lower()
            if 'phospho' in description:
                # Преобразуем позицию в число
                position_str = feature.get('begin')
                try:
                    position = int(position_str) if position_str else 0
                except (TypeError, ValueError):
                    position = 0
                
                # Исправленная проверка evidences
                evidence = 'N/A'
                if feature.get('evidences') and len(feature['evidences']) > 0:
                    evidence = feature['evidences'][0].get('source', {}).get('name', 'N/A')
                
                phosphorylation_sites.append({
                    'position': position,
                    'description': feature.get('description', ''),
                    'evidence': evidence
                })
    
    # Сортируем по позиции
    phosphorylation_sites.sort(key=lambda x: x['position'])
    
    print(f"Всего сайтов фосфорилирования: {len(phosphorylation_sites)}")
    
    # Группируем по ключевым регионам
    regions = {
        'N-концевой домен': [],
        'Pro-rich домен': [],
        'Microtubule-binding повторы': [],
        'C-концевой домен': []
    }
    
    for site in phosphorylation_sites:
        pos = site['position']
        if pos < 150:
            regions['N-концевой домен'].append(site)
        elif 150 <= pos < 250:
            regions['Pro-rich домен'].append(site)
        elif 250 <= pos < 400:
            regions['Microtubule-binding повторы'].append(site)
        else:
            regions['C-концевой домен'].append(site)
    
    for region, sites in regions.items():
        if sites:
            print(f"\n{region} ({len(sites)} сайтов):")
            for site in sites[:3]:  # Показываем по 3 сайта из каждого региона
                print(f"  Позиция {site['position']}: {site['description']}")

else:
    print(f"Ошибка запроса: {response.status_code}")

Сайты фосфорилирования Tau-белка:
Всего сайтов фосфорилирования: 42

N-концевой домен (7 сайтов):
  Позиция 18: Phosphotyrosine; by FYN
  Позиция 29: Phosphotyrosine
  Позиция 46: Phosphoserine

Pro-rich домен (1 сайтов):
  Позиция 214: Phosphoserine; by SGK1

C-концевой домен (34 сайтов):
  Позиция 470: Phosphothreonine; by PDPK1
  Позиция 486: Phosphothreonine
  Позиция 492: Phosphothreonine


Tau демонстрирует выраженную асимметрию фосфорилирования: C-концевой домен содержит 81% всех сайтов модификации (34 из 42), что указывает на его ключевую роль в регуляции функции белка. При этом N-концевой домен показывает специфическое тирозиновое фосфорилирование (Tyr18, Tyr29) через киназу FYN, связанную с синаптической передачей.

Такой паттерн фосфорилирования соответствует известным механизмам патологии: гиперфосфорилирование C-конца нарушает связывание с микротрубочками и способствует агрегации Tau, в то время как N-концевые модификации могут влиять на его синаптические функции и взаимодействие с мембранами.

Запрос 3: Поиск исследований о патологических формах Tau

In [ ]:
import requests
import xml.etree.ElementTree as ET

search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
search_params = {
    'db': 'pubmed',
    'term': 'tau protein AND (phosphorylation OR oligomers OR "neurofibrillary tangles") AND 2024[PDAT]',
    'retmode': 'json',
    'retmax': 8,
    'sort': 'relevance'
}

response = requests.get(search_url, params=search_params)

if response.status_code == 200:
    data = response.json()
    article_ids = data.get('esearchresult', {}).get('idlist', [])
    
    print(f"\nПоследние исследования патологии Tau (2024):")
    print("=" * 70)
    
    if article_ids:
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            'db': 'pubmed',
            'id': ','.join(article_ids[:5]),  # Берем первые 5
            'retmode': 'xml'
        }
        
        fetch_response = requests.get(fetch_url, params=fetch_params)
        if fetch_response.status_code == 200:
            root = ET.fromstring(fetch_response.text)
            
            for i, article in enumerate(root.findall('.//PubmedArticle')):
                title_elem = article.find('.//ArticleTitle')
                abstract_elem = article.find('.//AbstractText')
                date_elem = article.find('.//PubDate/Year')
                
                title = title_elem.text if title_elem is not None else "Нет заголовка"
                abstract = abstract_elem.text[:150] + "..." if abstract_elem is not None and abstract_elem.text else "Нет абстракта"
                date = date_elem.text if date_elem is not None else "2024"
                
                print(f"\n{i+1}. {title}")
                print(f"   Абстракт: {abstract}")
                print(f"   Год: {date}")


Последние исследования патологии Tau (2024):

1. TYK2 regulates tau levels, phosphorylation and aggregation in a tauopathy mouse model.
   Абстракт: Alzheimer's disease is one of at least 26 diseases characterized by tau-positive accumulation in neurons, glia or both. However, it is still unclear w...
   Год: 2024

2. P-tau217 correlates with neurodegeneration in Alzheimer's disease, and targeting p-tau217 with immunotherapy ameliorates murine tauopathy.
   Абстракт: Neuronal loss is the central issue in Alzheimer's disease (AD), yet no treatment developed so far can halt AD-associated neurodegeneration. Here, we d...
   Год: 2024

3. Blood Biomarkers to Detect Alzheimer Disease in Primary Care and Secondary Care.
   Абстракт: An accurate blood test for Alzheimer disease (AD) could streamline the diagnostic workup and treatment of AD....
   Год: 2024

4. Tau filaments are tethered within brain extracellular vesicles in Alzheimer's disease.
   Абстракт: The abnormal assembly of tau pro

Основной тренд - переход от фундаментальных механизмов к клиническим применениям: фосфорилированный Tau (p-tau217) подтвержден как ключевой биомаркер для диагностики, а иммунотерапия против него демонстрирует терапевтический потенциал в доклинических моделях.

При этом продолжают открываться новые механизмы патологии - роль TYK2 киназы в регуляции Tau и обнаружение Tau-филаментов во внеклеточных везикулах, что расширяет понимание распространения патологии между клетками.